# 77 — Q-planning fork pilot: manifest and restoration preflight

This notebook freezes 64 roots for each of three acquisition strategies (random, U20, and
failure), then audits one root per strategy against the **original persisted source**.
The sentinel measures replay-only drift, then snaps to the saved MuJoCo state, checks full and
root-suffix outcomes, and compares regenerated chunks with the source policy tensors.
It also renders one source-vs-collector continuation pair. Collection workers should not
be launched unless all three corrected source-fidelity checks pass. A nonzero pre-snap replay
error is diagnostic; the exact persisted-state snap is the restoration used by the workers.

The future training contract is **65% fork-priority / 35% ordinary replay**. This notebook
collects no training branches and does not touch the held-out position-perturbation suites.


## 1. Environment


In [ ]:
EXTRAS = 'sim,analysis'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen(
    'https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/'
    'pnp-vla/scripts/colab_bootstrap.py').read().decode())


## 2. Build and publish the immutable root manifest


In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.qplanning_fork_pilot import (
    FORK_PILOT_MANIFEST_PATH,
    create_fork_pilot_manifest,
)
from pnp.store import SupabaseStore

drive.mount('/content/drive')
store = SupabaseStore()
CACHE_ROOT = Path('/content/drive/MyDrive/pnp_qplanning_forks')

document = create_fork_pilot_manifest(
    store=store,
    cache_root=CACHE_ROOT,
    download_workers=8,
)
print({
    'manifest_path': FORK_PILOT_MANIFEST_PATH,
    'manifest_hash': document['manifest_hash'],
    'trees': len(document['payload']['trees']),
    'strategies': document['payload']['strategies'],
})


## 3. Strict source-fidelity sentinel


In [ ]:
from pnp.qplanning_fork_pilot import run_fork_source_fidelity_preflight

fidelity = run_fork_source_fidelity_preflight(
    manifest_path=FORK_PILOT_MANIFEST_PATH,
    store=store,
    video_dir=CACHE_ROOT / 'source_fidelity_videos',
)
restoration_report = fidelity['reports']
restoration_report


## 4. Visual source/continuation comparison

Both videos begin from the exact persisted source state and execute the same first 10 source
actions. The left side keeps the stored source suffix; the right side then switches to the
collector's newly sampled closed-loop continuation. This isolates continuation divergence
from restoration error.


In [ ]:
import base64
from html import escape
from IPython.display import HTML, display

video = fidelity['video']
def embedded_video(path):
    payload = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    return f"<video controls loop width='100%' src='data:video/mp4;base64,{payload}'></video>"

display(HTML(
    "<div style='display:grid;grid-template-columns:1fr 1fr;gap:16px'>"
    + f"<div><b>{escape(video['left_label'])}</b><br>outcome: {'S' if video['left_success'] else 'F'}"
    + embedded_video(video['left_path']) + "</div>"
    + f"<div><b>{escape(video['right_label'])}</b><br>outcome: {'S' if video['right_success'] else 'F'}"
    + embedded_video(video['right_path']) + "</div></div>"
))


## 5. Final stop/go assertion

This assertion is intentionally last so a failure does not hide the diagnostic table or videos.


In [ ]:
failed = [row for row in restoration_report if not row['passed']]
assert not failed, (
    'DO NOT launch/resume fork collectors: corrected persisted-source restoration failed. ',
    failed,
)
print('PASS: all three roots match their persisted source state and behavioral outcome.')
